# CV That Works: Step 9 evaluation notebook
Runs on your own computer only. It reads the file you export from `/api/review/export`, which holds real CVs you have corrected.
**Keep that file out of GitHub.** Never use made-up CVs here.

How to use: put this notebook in the `train` folder of the repo, save the export as `train/export.json`, then run each cell from the top.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))     # repo root, so we use the same checker as the live tool
from core.checker import check_cv
from core.rubric import MAX_POINTS

EXPORT_FILE = "export.json"
MIN_CVS = 50                                  # below this the notebook will not recommend the model
JUDGED = ["bullets", "summary", "language", "fit"]

## 1. Load the corrected CVs

In [ ]:
raw = json.loads(Path(EXPORT_FILE).read_text(encoding="utf-8"))
items = raw if isinstance(raw, list) else (raw.get("items") or raw.get("records") or raw.get("cvs") or [])
print(len(items), "corrected CVs in the file")
if items:
    print("Fields in the first record:", list(items[0].keys()))

## 2. Rebuild the rules counts from each CV's text
Page count is not stored, so it is estimated from word count. The model scores judged items only, so this matters little.

In [ ]:
def est_pages(text):
    return max(1, round(len(text.split()) / 450))

rows = []
for it in items:
    ts = (it.get("trainer") or {}).get("scores") or it.get("trainer_scores")
    if not ts or not it.get("text"):
        continue
    facts = check_cv(it["text"], est_pages(it["text"]), True, None)
    row = {"id": it.get("id"), "target": it.get("target")}
    row.update({"f_" + k: float(v) for k, v in facts.items() if isinstance(v, (int, float, bool))})
    cats = it.get("categories", {})
    for c in JUDGED:
        if c in ts:
            cc = cats.get(c, {})
            row["trainer_" + c] = ts[c]
            row["rules_" + c] = cc.get("rules")
            row["final_" + c] = cc.get("final")
    rows.append(row)
df = pd.DataFrame(rows)
print(len(df), "CVs usable")
feat_cols = [c for c in df.columns if c.startswith("f_")]
print(len(feat_cols), "rules features")

## 3. Test the model against simple guesses
Repeated 5-fold testing: each CV is scored by a model that never saw it. Error is the average gap from **your** score, in points.
Compared against: always guessing the average, the rules-only score, and the blended score the tool shows today.

In [ ]:
results = []
for c in JUDGED:
    need = ["trainer_" + c, "rules_" + c, "final_" + c]
    if not all(n in df.columns for n in need):
        continue
    d = df.dropna(subset=need)
    if len(d) < 10:
        print(f"{c}: only {len(d)} usable CVs, skipped")
        continue
    X, y = d[feat_cols].fillna(0).values, d["trainer_" + c].values
    errs = {"average": [], "rules": [], "blended": [], "model": []}
    for tr, te in RepeatedKFold(n_splits=5, n_repeats=5, random_state=7).split(X):
        m = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(X[tr], y[tr])
        pred = np.clip(m.predict(X[te]), 0, MAX_POINTS[c])
        errs["model"].append(np.abs(pred - y[te]).mean())
        errs["average"].append(np.abs(y[tr].mean() - y[te]).mean())
        errs["rules"].append(np.abs(d["rules_" + c].values[te] - y[te]).mean())
        errs["blended"].append(np.abs(d["final_" + c].values[te] - y[te]).mean())
    results.append({"category": c, "max_points": MAX_POINTS[c], "n": len(d), **{k: round(float(np.mean(v)), 2) for k, v in errs.items()}})
res = pd.DataFrame(results)
res

## 4. Verdict

In [ ]:
base = ["average", "rules", "blended"]
res["best_baseline"] = res[base].min(axis=1)
res["model_wins"] = res["model"] < res["best_baseline"]
print(res[["category", "n", "model", "best_baseline", "model_wins"]].to_string(index=False))
n = len(df)
wins, total = int(res["model_wins"].sum()), len(res)
better_overall = res["model"].mean() < res["best_baseline"].mean()
if n < MIN_CVS:
    ADD_MODEL = False
    print(f"\nVERDICT: do not add. Only {n} corrected CVs, and {MIN_CVS} is the minimum. Keep collecting.")
elif wins >= max(1, total - 1) and better_overall:
    ADD_MODEL = True
    print(f"\nVERDICT: the model beat the best simple guess in {wins} of {total} categories. It can join as the third voter.")
    if n < 100:
        print("Caution: under 100 corrected CVs. Treat it as an experiment and re-test as more arrive.")
else:
    ADD_MODEL = False
    print(f"\nVERDICT: do not add. The model beat the best simple guess in only {wins} of {total} categories.")

## 5. Save the model (only runs if the verdict is to add it)
The file is small (numbers only), so it fits the Vercel free plan with no extra packages.

In [ ]:
if ADD_MODEL:
    out = {"version": "model-v1", "rubric_version": "cv-rubric-v1.0", "features": feat_cols, "categories": {}}
    for c in res["category"]:
        d = df.dropna(subset=["trainer_" + c])
        X, y = d[feat_cols].fillna(0).values, d["trainer_" + c].values
        pipe = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(X, y)
        sc, rg = pipe.named_steps["standardscaler"], pipe.named_steps["ridge"]
        out["categories"][c] = {"max": MAX_POINTS[c], "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(),
                                "coef": rg.coef_.tolist(), "intercept": float(rg.intercept_)}
    Path("model_v1.json").write_text(json.dumps(out))
    print("Saved model_v1.json")
else:
    print("Nothing saved.")